<a href="https://colab.research.google.com/github/lucianoselimaj/SigExt/blob/MMD/notebooks/hallucination_filter_extension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title requirements
!pip install sentence-transformers jsonlines rouge_score transformers mistralai -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
#@title Confirm that the shared results directory exists
#@markdown Verify the directory path on your Google Drive, and change it if needed
import os
shared_path = "/content/drive/MyDrive/ColabNotebooks/SigExt/experiments" #@param{type:"string"}
print(os.path.exists(shared_path))

True


In [ ]:
#@title Check if the required directories are present
base_path = shared_path

# check all needed files exist
files_to_check = [
    f"{base_path}/cnn_dataset_with_keyphrase/test.jsonl",
    f"{base_path}/cnn_extsig_predictions_mistral-small-2603_k15/test_predictions.json",
    f"{base_path}/cnn_extsig_predictions_mistral-small-2603_k15/test_dataset.jsonl",
    f"{base_path}/cnn_extsig_predictions_mistral-small-2603_k15/test_metrics.json",
]

for f in files_to_check:
  exists = os.path.exists(f)
  print(f"{'✓' if exists else '✗'}  {f}")

✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_dataset_with_keyphrase/test.jsonl
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15/test_predictions.json
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15/test_dataset.jsonl
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15/test_metrics.json


#Clone the repo

In [ ]:
# @markdown check the check-box to clone the repo from sorce, <b>otherwise it will be loaded from shared google drive files</b>

clone_repo = False # @param {type:"boolean"}
if clone_repo:
  !git clone https://github.com/lucianoselimaj/SigExt.git
  path = "/SigExt"

else:
  path = "/content/drive/MyDrive/DNLP-storage/SigExt"

# Preparing Data & Pipelines

In [ ]:
# @title Download requirements for data preparing

%cd {shared_path}

import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

/content/drive/.shortcut-targets-by-id/1TpffEVc0xhfd78UxnDSQbY7er61_yzCk/SigExt/experiments


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import os
import json
import nltk #imports the Natural Language Toolkit used to split text into sentences
import torch
import numpy as np
import tqdm #progress bar library
import jsonlines
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util #util provides the cosine similarity
from transformers import pipeline as hf_pipeline #provides AI models

#define all paths
base = shared_path
path_scored_dataset   = f"{base}/cnn_dataset_with_keyphrase/test.jsonl" #Path to the test articles that have phrase scores added by the Longformer
path_predictions      = f"{base}/cnn_extsig_predictions_mistral-small-2603_k15/test_predictions.json" #Path to the 500 raw summaries generated by Mistral
path_full_dataset     = f"{base}/cnn_extsig_predictions_mistral-small-2603_k15/test_dataset.jsonl" #Path to the full dataset file that includes raw_output
path_baseline_metrics = f"{base}/cnn_extsig_predictions_mistral-small-2603_k15/test_metrics.json" #Path to the ROUGE scores that were computed by the original SigExt pipeline
path_output           = f"{base}/hallucination_extension/cnn_verified_predictions/" #Path to the folder where the extension will save its results

os.makedirs(path_output, exist_ok=True)

TOP_N_EVIDENCE     = 5 #Controls how many source sentences are retrieved as evidence for each summary sentence
ENTAIL_THRESHOLD   = 0.5 #The minimum confidence score the NLI model must assign to "entailment" for a sentence to pass
MAX_REGEN_ATTEMPTS = 2 #How many times the extension tries to regenerate a flagged sentence before giving up and dropping it
SIMILARITY_THRESHOLD = 0.75

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


Device: cuda


In [ ]:
#@title Load Models
print("Loading sentence embedder...")
embedder = SentenceTransformer("all-MiniLM-L6-v2") #Downloads and loads the sentence embedding model, used to produce meaningful sentence embeddings
embedder = embedder.to(device)

print("Loading NLI model...")
#Instead of manually tokenizing input, running the model, and decoding output, the pipeline handles everything automatically
nli_model = hf_pipeline(
    "text-classification",
    model="dleemiller/ModernCE-large-nli",
    device=0 if device == "cuda" else -1,
    top_k=None,
    batch_size=16,
    truncation=True,
    max_length=512,
)


Loading sentence embedder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [ ]:
import sys
import os
from types import ModuleType
from mistralai.client import Mistral
import tqdm

MISTRAL_API_KEY = "YdbLhqEepgXav0x8ruTJiWR20GsgWCrD"


def predict_one_eg_mistral(example): #Translte phrases for mistral
    client = Mistral(api_key=MISTRAL_API_KEY)
    try:
        chat_response = client.chat.complete(
            model="mistral-small-latest",
            messages=[{"role": "user", "content": example["prompt_input"]}]
        )
        return chat_response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

In [ ]:
#@title Load Data

#raw summaries generated by the LLMs
with open(path_predictions) as f:
  raw_summaries = json.load(f)

#source articles with phrase scores
with jsonlines.open(path_scored_dataset) as f:
  scored_dataset = list(f)

#full dataset with raw_output for the ROUGE evaluation
with jsonlines.open(path_full_dataset) as f:
  full_dataset = list(f)

#baseline metrics for comparison
with open(path_baseline_metrics) as f:
  baseline_metrics = json.load(f)

print(f"Summaries loaded:       {len(raw_summaries)}")
print(f"Scored articles loaded: {len(scored_dataset)}")
print(f"Full dataset loaded:    {len(full_dataset)}")
print(f"\nBaseline ROUGE-1 F1: {baseline_metrics.get('rouge1f', 'N/A')}")
print(f"\nSample raw summary:\n{raw_summaries[0]}")

Summaries loaded:       500
Scored articles loaded: 500
Full dataset loaded:    500

Baseline ROUGE-1 F1: 38.8941

Sample raw summary:
A former Japanese school principal, Yuhei Takashima, is facing criminal charges in Tokyo after police arrested him for allegedly paying for sex with over 12,000 women—some as young as 14—in the Philippines, where he also photographed obscene acts and produced pornography. Police seized 147,600 photos documenting his activities, with investigations revealing he traveled to Manila repeatedly since 1988 to "buy sex," citing its affordability.


#Verification Functions

In [ ]:
#@title Verification Functions

import re
import nltk
import torch
from sentence_transformers import util

# ── Stage 4 — sentence segmentation ──────────────────────────
def split_sentences(text):
    if not text or not text.strip():
        return []
    return [
        s.strip()
        for s in nltk.sent_tokenize(text)
        if s.strip() and len(s.split()) >= 3
    ]


# ── Stage 5 — evidence retrieval ─────────────────────────────
def retrieve_evidence(summary_sentence, source_sentences, top_n=TOP_N_EVIDENCE):
    """
    Retrieves the most semantically similar source sentences as evidence.
    Uses cosine similarity between sentence embeddings.
    Returns top_n most similar source sentences.
    """
    query_emb  = embedder.encode(summary_sentence, convert_to_tensor=True)
    source_emb = embedder.encode(source_sentences, convert_to_tensor=True)
    scores     = util.cos_sim(query_emb, source_emb)
    top_idx    = torch.topk(
        scores, k=min(top_n, len(source_sentences))
    ).indices.flatten()
    return [source_sentences[idx.item()] for idx in top_idx]


# ── Stage 6 — hybrid entailment check ────────────────────────
def check_entailment(summary_sentence, evidence):
    """
    Hybrid check combining semantic similarity and NLI.

    A sentence passes if:
    1. NLI entailment score >= threshold for ANY evidence sentence
    OR
    2. Semantic similarity >= similarity threshold for ANY evidence sentence
       (catches paraphrases that NLI misses)

    A sentence is only flagged as hallucination if:
    - Both NLI and similarity fail
    - AND NLI returns contradiction (not just neutral)
    """
    best_entail  = 0.0
    best_contra  = 0.0
    best_neutral = 0.0
    best_sim     = 0.0

    # encode summary sentence once
    summary_emb = embedder.encode(summary_sentence, convert_to_tensor=True)

    for ev_sentence in evidence:

        # compute semantic similarity
        ev_emb = embedder.encode(ev_sentence, convert_to_tensor=True)
        sim    = float(util.cos_sim(summary_emb, ev_emb)[0][0])
        if sim > best_sim:
            best_sim = sim

        # compute NLI
        result = nli_model({
            "text":      ev_sentence,
            "text_pair": summary_sentence
        })
        label_scores = {item["label"]: item["score"] for item in result}

        entail  = label_scores.get("entailment",    0.0)
        contra  = label_scores.get("contradiction", 0.0)
        neutral = label_scores.get("neutral",       0.0)

        if entail  > best_entail:  best_entail  = entail
        if contra  > best_contra:  best_contra  = contra
        if neutral > best_neutral: best_neutral = neutral

    # pass conditions:
    # 1. NLI says entailed
    # 2. OR sentences are very semantically similar (paraphrase)
    if best_entail >= ENTAIL_THRESHOLD or best_sim >= SIMILARITY_THRESHOLD:
        return "entailed", max(best_entail, best_sim)

    # fail conditions — only flag real hallucinations
    elif best_contra >= 0.7:
        return "contradiction", best_contra
    else:
        # neutral with low similarity — invented information
        return "neutral", best_neutral

# ── Stage 7 — sentence regeneration ──────────────────────────
def regenerate_sentence(
    original_sentence,
    evidence,
    predict_fn=predict_one_eg_mistral
):
    """
    Asks the LLM to rewrite a flagged sentence using only the evidence.
    Returns None if the LLM cannot produce a faithful rewrite.
    Keyphrases are NOT passed here to avoid reproducing the same error.
    """
    evidence_text = " ".join(evidence)
    prompt = (
        f"The following sentence may contain information not supported "
        f"by the source document.\n\n"
        f"Evidence from the source:\n{evidence_text}\n\n"
        f"Original sentence:\n{original_sentence}\n\n"
        f"Rewrite the sentence using only information from the evidence. "
        f"Do not add anything not in the evidence. "
        f'If impossible, respond with exactly "UNSUPPORTED".'
    )
    result = predict_fn({"prompt_input": prompt})
    if not result or result.strip().upper() == "UNSUPPORTED":
        return None
    return result.strip()


# ── Stages 4-8 — pipeline orchestrator ───────────────────────
def verify_summary(
    raw_summary,
    source_document,
    predict_fn=predict_one_eg_mistral
):
    """
    Orchestrates the full verification pipeline for one summary.

    For each summary sentence:
        1. Retrieve top-N most similar source sentences as evidence
        2. Check pairwise NLI entailment against each evidence sentence
        3. If passes → keep sentence unchanged
        4. If fails  → attempt regeneration using evidence as constraint
        5. If regeneration passes NLI → use regenerated sentence
        6. If regeneration fails     → drop sentence

    Returns the verified summary and a log of all hallucinations found.
    """
    summary_sentences = split_sentences(raw_summary)
    source_sentences  = split_sentences(source_document)

    if not summary_sentences or not source_sentences:
        return raw_summary, []

    verified   = []
    halluc_log = []

    for i, sentence in enumerate(summary_sentences):

        # Stage 5 — retrieve evidence
        evidence = retrieve_evidence(sentence, source_sentences)

        # Stage 6 — pairwise entailment check
        label, score = check_entailment(sentence, evidence)

        # sentence passes — keep it unchanged
        if label == "entailed":
            verified.append(sentence)
            continue

        # sentence flagged — create audit log entry
        log_entry = {
            "sentence_idx":   i,
            "original":       sentence,
            "label":          label,
            "score":          round(score, 4),
            "evidence":       evidence,
            "regenerated":    None,
            "regen_accepted": False,
        }

        # Stage 7 — attempt regeneration
        regenerated = None
        for attempt in range(MAX_REGEN_ATTEMPTS):
            candidate = regenerate_sentence(sentence, evidence, predict_fn)

            if candidate is None:
                break

            # verify the regenerated sentence with pairwise NLI
            new_label, new_score = check_entailment(candidate, evidence)
            if new_label == "entailed":
                regenerated = candidate
                log_entry["regen_accepted"] = True
                break

        # Stage 8 — finalize log and add to verified if successful
        log_entry["regenerated"] = regenerated
        halluc_log.append(log_entry)

        if regenerated:
            verified.append(regenerated)

    # reassemble final summary
    final_summary = " ".join(verified)
    return final_summary, halluc_log


print("Verification functions defined.")

Verification functions defined.


#Tuning

In [ ]:
#@title Hyperparameter Tuning (Optuna Bayesian Optimization)
!pip install optuna -q

import optuna
from sklearn.model_selection import ShuffleSplit
from rouge_score import rouge_scorer
import numpy as np
import json
import os

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── tuning config ─────────────────────────────────────────────
# 80/20 split: 400 tune / 100 val — never touches the final test logic
TUNE_SIZE = 500
N_TRIALS  = 40  # Optuna evaluates 40 smart trials instead of 160 blind ones

ss = ShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tune_idx, val_idx = next(ss.split(range(TUNE_SIZE)))

tune_summaries = [raw_summaries[i]  for i in tune_idx]
tune_dataset   = [scored_dataset[i] for i in tune_idx]

val_summaries  = [raw_summaries[i]  for i in val_idx]
val_dataset    = [scored_dataset[i] for i in val_idx]
val_refs       = [full_dataset[i]["raw_output"] for i in val_idx]


# ── evaluation function ───────────────────────────────────────
def evaluate_config(top_n, threshold, similarity, max_regen):
    global TOP_N_EVIDENCE, ENTAIL_THRESHOLD, SIMILARITY_THRESHOLD, MAX_REGEN_ATTEMPTS
    TOP_N_EVIDENCE       = top_n
    ENTAIL_THRESHOLD     = threshold
    SIMILARITY_THRESHOLD = similarity
    MAX_REGEN_ATTEMPTS   = max_regen

    final_sums = []
    logs       = []

    for raw_sum, article in zip(val_summaries, val_dataset):
        if not raw_sum or not raw_sum.strip():
            final_sums.append("")
            logs.append([])
            continue
        src = article["trunc_input"]
        final_sum, halluc_log = verify_summary(raw_sum, src)
        final_sums.append(final_sum)
        logs.append(halluc_log)

    # ── ROUGE-1 and ROUGE-L ───────────────────────────────────
    scorer_obj = rouge_scorer.RougeScorer(
        ["rouge1", "rougeL"], use_stemmer=True
    )
    r1_f, rl_f, r1_r = [], [], []
    for pred, ref in zip(final_sums, val_refs):
        if not pred or not pred.strip():
            pred = "empty"
        score = scorer_obj.score(target=ref, prediction=pred)
        r1_f.append(score["rouge1"].fmeasure)
        rl_f.append(score["rougeL"].fmeasure)
        r1_r.append(score["rouge1"].recall)

    rouge1f = round(np.mean(r1_f) * 100, 4)
    rougeLf = round(np.mean(rl_f) * 100, 4)
    rouge1r = round(np.mean(r1_r) * 100, 4)

    # ── hallucination statistics ──────────────────────────────
    total_sents   = sum(len(split_sentences(s)) for s in val_summaries if s)
    total_flagged = sum(len(l) for l in logs)
    halluc_rate   = total_flagged / total_sents if total_sents > 0 else 0

    # ── regeneration success rate ─────────────────────────────
    regen_ok = sum(
        sum(1 for h in log if h["regen_accepted"])
        for log in logs
    )
    regen_rate = regen_ok / total_flagged if total_flagged > 0 else 0

    # ── empty summaries & dropped sentences ──────────────────
    empty   = sum(1 for s in final_sums if not s or not s.strip())
    dropped = total_flagged - regen_ok

    return {
        "rouge1f":       rouge1f,
        "rougeLf":       rougeLf,
        "rouge1r":       rouge1r,
        "halluc_rate":   round(halluc_rate, 4),
        "regen_rate":    round(regen_rate, 4),
        "total_flagged": total_flagged,
        "regen_ok":      regen_ok,
        "dropped":       dropped,
        "empty":         empty,
    }


# ── Optuna objective ──────────────────────────────────────────
tuning_results = []

def objective(trial):
    top_n      = trial.suggest_categorical("top_n",      [3, 5, 7, 10])
    threshold  = trial.suggest_categorical("threshold",  [0.2, 0.3, 0.4, 0.5, 0.6])
    similarity = trial.suggest_categorical("similarity", [0.55, 0.65, 0.75, 0.85])
    max_regen  = trial.suggest_categorical("max_regen",  [1, 2])

    metrics = evaluate_config(top_n, threshold, similarity, max_regen)

    tuning_results.append({
        "top_n":      top_n,
        "threshold":  threshold,
        "similarity": similarity,
        "max_regen":  max_regen,
        **metrics
    })

    return metrics["rouge1f"]  # Optuna maximizes this


# ── run Optuna study ──────────────────────────────────────────
print(f"Running Optuna with {N_TRIALS} trials on {len(val_summaries)} validation examples...")
print(f"Estimated time on A100: ~{N_TRIALS * len(val_summaries) * 0.3 / 60:.0f} minutes\n")

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),  # Tree-structured Parzen Estimator
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)


# ── sort and print top 15 ─────────────────────────────────────
tuning_results.sort(key=lambda x: x["rouge1f"], reverse=True)

print(f"\n{'top_n':<8} {'thresh':<8} {'sim':<8} {'regen':<8} "
      f"{'R1-F1':<10} {'RL-F1':<10} {'halluc%':<10} "
      f"{'regen%':<8} {'dropped':<8} {'empty':<6}")
print("-" * 84)
for r in tuning_results[:15]:
    print(
        f"{r['top_n']:<8} {r['threshold']:<8} {r['similarity']:<8} "
        f"{r['max_regen']:<8} {r['rouge1f']:<10} {r['rougeLf']:<10} "
        f"{r['halluc_rate']*100:<10.1f} {r['regen_rate']*100:<8.1f} "
        f"{r['dropped']:<8} {r['empty']:<6}"
    )


# ── apply best config ─────────────────────────────────────────
best = study.best_params
TOP_N_EVIDENCE       = best["top_n"]
ENTAIL_THRESHOLD     = best["threshold"]
SIMILARITY_THRESHOLD = best["similarity"]
MAX_REGEN_ATTEMPTS   = best["max_regen"]

best_metrics = tuning_results[0]
print(f"\nBest configuration applied:")
print(f"  TOP_N_EVIDENCE       = {TOP_N_EVIDENCE}")
print(f"  ENTAIL_THRESHOLD     = {ENTAIL_THRESHOLD}")
print(f"  SIMILARITY_THRESHOLD = {SIMILARITY_THRESHOLD}")
print(f"  MAX_REGEN_ATTEMPTS   = {MAX_REGEN_ATTEMPTS}")
print(f"  ROUGE-1 F1           = {best_metrics['rouge1f']}")
print(f"  ROUGE-L F1           = {best_metrics['rougeLf']}")
print(f"  Hallucination rate   = {best_metrics['halluc_rate']*100:.1f}%")
print(f"  Regenerated accepted = {best_metrics['regen_ok']}")
print(f"  Dropped              = {best_metrics['dropped']}")
print(f"  Empty summaries      = {best_metrics['empty']}")


# ── save all tuning results ───────────────────────────────────
os.makedirs(path_output, exist_ok=True)
with open(f"{path_output}tuning_results_optuna.json", "w") as f:
    json.dump(tuning_results, f, indent=2)
print(f"\nAll {len(tuning_results)} results saved to "
      f"{path_output}tuning_results_optuna.json")

Testing 72 combinations on 50 examples...


Tuning: 100%|██████████| 72/72 [1:21:30<00:00, 67.92s/it]


top_n    thresh   sim      regen    ROUGE-1    halluc%    regen%   empty 
--------------------------------------------------------------------
3        0.3      0.65     1        39.2666    10.9       45.5     0     
3        0.3      0.65     2        39.0947    10.9       54.5     0     
7        0.3      0.75     2        39.0931    30.7       61.3     0     
7        0.5      0.65     2        39.0806    12.9       38.5     0     
5        0.2      0.65     1        39.0389    8.9        33.3     0     
5        0.2      0.65     2        39.0354    8.9        44.4     0     
3        0.2      0.75     1        38.9975    26.7       59.3     0     
5        0.5      0.75     2        38.9787    34.6       62.9     0     
5        0.3      0.75     1        38.9655    30.7       54.8     0     
7        0.3      0.65     1        38.9256    10.9       45.5     0     

Best configuration applied:
  TOP_N_EVIDENCE       = 3
  ENTAIL_THRESHOLD     = 0.3
  SIMILARITY_THRESHOLD = 0.65
 

#Final Pipeline

In [ ]:
final_summaries = []
all_halluc_logs  = []


"""
At each iteration:
i = 0
raw_summary = "Scientists confirmed the Higgs boson. The cost was 13 billion."
article     = {
    "trunc_input": "Scientists at CERN confirmed the Higgs boson particle.
                    The research involved 6,000 scientists...",
    "input_kw_model": [...],
    ...
}
"""
for i, (raw_summary, article) in enumerate(
    tqdm.tqdm(
        zip(raw_summaries, scored_dataset),
        total=len(raw_summaries),
        desc="Verifying"
    )
):
    # skip empty summaries
    if not raw_summary or not raw_summary.strip():
        final_summaries.append("")
        all_halluc_logs.append({
            "example_idx":    i,
            "raw_summary":    raw_summary,
            "final_summary":  "",
            "hallucinations": [],
        })
        continue

    source_document = article["trunc_input"]
    final_summary, halluc_log = verify_summary(raw_summary, source_document)

    final_summaries.append(final_summary)
    all_halluc_logs.append({
        "example_idx":    i,
        "raw_summary":    raw_summary,
        "final_summary":  final_summary,
        "hallucinations": halluc_log,
    })

print(f"\nVerification complete.")
print(f"Total examples processed: {len(final_summaries)}")
print(f"Total hallucinations found: {sum(len(l['hallucinations']) for l in all_halluc_logs)}")

Verifying: 100%|██████████| 500/500 [04:42<00:00,  1.77it/s]


Verification complete.
Total examples processed: 500
Total hallucinations found: 132


In [ ]:
#@title Evaluate and save results (Stage 9)
from rouge_score import rouge_scorer
import numpy as np
import json
import os

def compute_rouge(predictions, references):
    scorer_obj = rouge_scorer.RougeScorer(
        ["rouge1", "rougeL"], use_stemmer=True
    )
    r1_f, rl_f, r1_r = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred or not pred.strip():
            pred = "empty"
        score = scorer_obj.score(target=ref, prediction=pred)
        r1_f.append(score["rouge1"].fmeasure)
        rl_f.append(score["rougeL"].fmeasure)
        r1_r.append(score["rouge1"].recall)
    return {
        "rouge1f": round(np.mean(r1_f) * 100, 4),
        "rougeLf": round(np.mean(rl_f) * 100, 4),
        "rouge1r": round(np.mean(r1_r) * 100, 4),
    }

# human reference summaries
references = [item["raw_output"] for item in full_dataset]

# compute ROUGE for raw and verified summaries
raw_metrics   = compute_rouge(raw_summaries,   references)
final_metrics = compute_rouge(final_summaries, references)

# hallucination statistics
total_sentences = sum(
    len(split_sentences(log["raw_summary"]))
    for log in all_halluc_logs
    if log["raw_summary"]
)
total_flagged = sum(
    len(log["hallucinations"])
    for log in all_halluc_logs
)
regen_accepted = sum(
    sum(1 for h in log["hallucinations"] if h["regen_accepted"])
    for log in all_halluc_logs
)
dropped      = total_flagged - regen_accepted
halluc_rate  = total_flagged / total_sentences if total_sentences > 0 else 0

# print results table
print("=" * 52)
print(f"{'Metric':<28} {'Baseline':>10} {'Verified':>10}")
print("=" * 52)
print(f"{'ROUGE-1 F1':<28} {raw_metrics['rouge1f']:>10} {final_metrics['rouge1f']:>10}")
print(f"{'ROUGE-L F1':<28} {raw_metrics['rougeLf']:>10} {final_metrics['rougeLf']:>10}")
print(f"{'ROUGE-1 Recall':<28} {raw_metrics['rouge1r']:>10} {final_metrics['rouge1r']:>10}")
print("=" * 52)
print(f"{'Total sentences':<28} {total_sentences:>21}")
print(f"{'Flagged sentences':<28} {total_flagged:>21}")
print(f"{'Hallucination rate':<28} {halluc_rate:>20.2%}")
print(f"{'Regenerated accepted':<28} {regen_accepted:>21}")
print(f"{'Dropped':<28} {dropped:>21}")
print("=" * 52)

# save results to Drive
results = {
    "raw_metrics":        raw_metrics,
    "verified_metrics":   final_metrics,
    "hallucination_rate": round(halluc_rate, 4),
    "total_sentences":    total_sentences,
    "total_flagged":      total_flagged,
    "regen_accepted":     regen_accepted,
    "dropped":            dropped,
}

os.makedirs(path_output, exist_ok=True)

with open(f"{path_output}metrics_verified.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nmetrics_verified.json saved")

with open(f"{path_output}halluc_log.json", "w") as f:
    json.dump(all_halluc_logs, f, indent=2)
print("halluc_log.json saved")

with open(f"{path_output}final_predictions.json", "w") as f:
    json.dump(final_summaries, f, indent=2)
print("final_predictions.json saved")

# verify files exist and show sizes
print("\nFiles saved:")
for fname in ["metrics_verified.json", "halluc_log.json", "final_predictions.json"]:
    fpath = f"{path_output}{fname}"
    size  = os.path.getsize(fpath) if os.path.exists(fpath) else 0
    print(f"  {fname}  ({size:,} bytes)")

Metric                         Baseline   Verified
ROUGE-1 F1                      38.8941    38.8691
ROUGE-L F1                      24.2784     24.444
ROUGE-1 Recall                  47.2585    45.9837
Total sentences                               1004
Flagged sentences                              132
Hallucination rate                         13.15%
Regenerated accepted                            80
Dropped                                         52

metrics_verified.json saved
halluc_log.json saved
final_predictions.json saved

Files saved:
  metrics_verified.json  (320 bytes)
  halluc_log.json  (598,459 bytes)
  final_predictions.json  (205,099 bytes)
